# Visual Genome Caption Generation
## Task 2 Training: Relationship Classification

Train relationship classifier for object pairs.

In [ ]:
# Imports
import os
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.utils import load_config, Logger, CheckpointManager
from src.data import RelationshipDataset, build_task2_datasets
from src.models.task2 import RelationClassifier
from src.training import Task2Trainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

In [ ]:
# Load configuration
config = load_config("../configs/task2_config.yaml")
print("Task 2 Configuration:")
print(f"- Relations: {config.labels.max_relations}")
print(f"- Spatial features: {config.model.use_spatial_features}")
print(f"- Fusion method: {config.model.fusion_method}")
print(f"- Batch size: {config.training.batch_size}")
print(f"- Max epochs: {config.training.max_epochs}")
print(f"- Learning rate: {config.training.lr}")

In [ ]:
# Create datasets
print("Creating Task 2 datasets...")

# Check if processed data exists
processed_dir = Path("../data/processed/task2")
if not processed_dir.exists():
    raise FileNotFoundError("Processed data not found. Run notebook 02_data_processing.ipynb first")

train_ds, val_ds, test_ds = build_task2_datasets(
    processed_dir=str(processed_dir),
    image_dir="../data/raw/images",
    roi_size=config.image.roi_size,
    use_feature_cache=config.dataset.use_feature_cache,
    use_spatial_features=config.model.use_spatial_features,
    max_samples=None  # Use all samples
)

print(f"Train: {len(train_ds)} samples")
print(f"Val: {len(val_ds)} samples")
print(f"Test: {len(test_ds)} samples")

# Create data loaders
train_loader = DataLoader(
    train_ds,
    batch_size=config.training.batch_size,
    shuffle=True,
    num_workers=config.training.num_workers,
    pin_memory=config.training.pin_memory
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.training.batch_size,
    shuffle=False,
    num_workers=config.training.num_workers,
    pin_memory=config.training.pin_memory
)

print("Data loaders created successfully")

In [ ]:
# Create model
print("Creating Task 2 model...")

device = config.device if torch.cuda.is_available() else "cpu"

model = RelationClassifier(
    num_relations=config.labels.max_relations,
    feature_dim=config.backbone.feature_dim,
    spatial_dim=9 if config.model.use_spatial_features else 0,
    hidden_dim=config.model.hidden_dim,
    dropout=config.model.dropout,
    fusion_method=config.model.fusion_method,
    device=device
)

print(f"Model: {model.summary()}")

# Create optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.training.lr,
    weight_decay=config.optimizer.weight_decay
)

print("Model and optimizer created successfully")

In [ ]:
# Setup logging and checkpointing
experiment_name = f"task2_{config.model.fusion_method}_{config.training.lr:.0e}"

logger = Logger(
    log_dir="../logs",
    experiment_name=experiment_name,
    use_tensorboard=True,
    use_wandb=False
)

checkpoint_manager = CheckpointManager(
    checkpoint_dir="../checkpoints",
    experiment_name=experiment_name,
    max_checkpoints=3,
    save_best_only=False
)

print(f"Experiment: {experiment_name}")

# Create trainer
trainer = Task2Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    logger=logger,
    checkpoint_manager=checkpoint_manager,
    max_epochs=config.training.max_epochs,
    early_stopping_patience=config.training.early_stopping_patience,
    gradient_clip_val=config.training.gradient_clip_val,
    log_every_n_steps=config.training.log_every_n_steps
)

print("Trainer created successfully")
print(f"Training for {config.training.max_epochs} epochs on {device}")

# Start training
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50)

final_metrics = trainer.train()

print("\n" + "="*50)
print("TRAINING COMPLETED")
print("="*50)
print("Final validation metrics:")
for key, value in final_metrics.items():
    print(".4f")

print("\nTraining notebook completed!")
print("Check ../logs/ and ../checkpoints/ for results")